# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


Python 실행 파일: /workspaces/llm-data-analysis-study/llm-data-analysis-course/.venv/bin/python
프로젝트 루트: /workspaces/llm-data-analysis-study/llm-data-analysis-course
전처리 데이터 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/data/processed
보고서 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


customers: (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders: (300, 7) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']
order_items: (764, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


In [ ]:
# 가입일이 주문일보다 늦은 주문을 찾아 원본 값을 확인합니다. (별도 추가 코드)
date_check = orders[
    ["order_id", "customer_id", "order_date"]
].merge(
    customers[["customer_id", "signup_date"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

date_check["order_date"] = pd.to_datetime(
    date_check["order_date"], errors="coerce"
)
date_check["signup_date"] = pd.to_datetime(
    date_check["signup_date"], errors="coerce"
)

invalid_signup_order = date_check.loc[
    date_check["signup_date"] > date_check["order_date"]
].copy()

print("가입일이 주문일보다 늦은 주문:", len(invalid_signup_order))
print("관련 고객 수:", invalid_signup_order["customer_id"].nunique())
display(
    invalid_signup_order.sort_values(
        ["customer_id", "order_date", "order_id"]
    )
)

가입일이 주문일보다 늦은 주문: 56
관련 고객 수: 31


,order_id,customer_id,order_date,signup_date,_merge
60,61,4,2025-07-19,2026-04-13,both
18,19,6,2025-11-25,2026-06-27,both
162,163,6,2026-03-12,2026-06-27,both
95,96,6,2026-05-11,2026-06-27,both
168,169,9,2025-07-12,2025-10-19,both
192,193,9,2025-09-03,2025-10-19,both
96,97,11,2025-07-14,2025-12-21,both
287,288,15,2025-08-25,2026-06-04,both
258,259,15,2026-05-11,2026-06-04,both
274,275,21,2025-09-01,2026-04-02,both


In [ ]:
# 56건을 주문 상태별로 집계해 함수의 타깃 범위와 비교합니다. (별도 추가 코드)
invalid_with_status = invalid_signup_order.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="left",
    validate="one_to_one",
)

display(
    invalid_with_status.groupby("order_status")
    .agg(
        invalid_order_count=("order_id", "size"),
        customer_count=("customer_id", "nunique"),
    )
    .reset_index()
)

# Chapter 10 분류 대상(completed, cancelled)만 확인합니다.
binary_invalid = invalid_with_status.loc[
    invalid_with_status["order_status"].isin(["completed", "cancelled"])
].copy()

print("전체 상태 포함:", len(invalid_with_status))
print("분류 대상 상태만:", len(binary_invalid))
print(
    "분류 대상 주문 상태:",
    binary_invalid["order_status"].value_counts().to_dict(),
)

,order_status,invalid_order_count,customer_count
0,cancelled,10,10
1,completed,35,23
2,refunded,11,9


전체 상태 포함: 56
분류 대상 상태만: 45
분류 대상 주문 상태: {'completed': 35, 'cancelled': 10}


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [5]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


,order_status,order_count
0,completed,184
1,cancelled,64
2,refunded,52


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [6]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


모델링 데이터: (248, 16)
숫자형 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup']
범주형 feature: ['gender', 'city', 'payment_method']


,is_cancelled,class_label,count,ratio
0,0,completed,184,0.7419
1,1,cancelled,64,0.2581


,merge,before_rows,after_rows,row_count_preserved,unmatched_count,status
0,order_id → order_item_features,248,248,True,0,PASS
1,customer_id → customers,248,248,True,0,PASS


,check_item,count
0,binary_target_rows,248
1,completed_rows,184
2,cancelled_rows,64
3,excluded_status_rows,52
4,line_total_mismatch,0
5,signup_after_order_days_set_missing,45
6,missing_order_item_features,0
7,missing_customer_match,0
8,status_scope:completed,184
9,status_scope:cancelled,64


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [7]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


사용 feature: ['age', 'item_count', 'total_quantity', 'order_amount', 'order_month', 'order_dayofweek', 'days_since_signup', 'gender', 'city', 'payment_method']
데이터 누수 컬럼 없음


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [8]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


X_train: (148, 10)
X_valid: (50, 10)
X_test: (50, 10)


,split,is_cancelled,class_label,count,ratio
0,train,0,completed,110,0.7432
1,train,1,cancelled,38,0.2568
2,validation,0,completed,37,0.7400
3,validation,1,cancelled,13,0.2600
4,test,0,completed,37,0.7400
5,test,1,cancelled,13,0.2600


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [9]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


,model,evaluation_split,accuracy,precision,recall,f1
0,Logistic Regression,validation,0.66,0.388889,0.538462,0.451613
1,Random Forest,validation,0.66,0.300000,0.230769,0.260870
2,Dummy Most Frequent,validation,0.74,0.000000,0.000000,0.000000


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [10]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


선택 모델: Logistic Regression
선택 임계값: 0.2


,threshold,accuracy,precision,recall,f1
0,0.20,0.38,0.295455,1.000000,0.456140
9,0.65,0.76,0.555556,0.384615,0.454545
6,0.50,0.66,0.388889,0.538462,0.451613
1,0.25,0.40,0.292683,0.923077,0.444444
4,0.40,0.58,0.333333,0.615385,0.432432
7,0.55,0.68,0.400000,0.461538,0.428571
3,0.35,0.50,0.300000,0.692308,0.418605
2,0.30,0.44,0.285714,0.769231,0.416667
5,0.45,0.60,0.333333,0.538462,0.411765
8,0.60,0.70,0.416667,0.384615,0.400000


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [11]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.2,frozen_before_test,0.28,0.244444,0.846154,0.37931


,pred_completed,pred_cancelled
actual_completed,3,34
actual_cancelled,2,11


,label,precision,recall,f1-score,support
0,completed,0.600000,0.081081,0.142857,37.00
1,cancelled,0.244444,0.846154,0.379310,13.00
2,accuracy,0.280000,0.280000,0.280000,0.28
3,macro avg,0.422222,0.463617,0.261084,50.00
4,weighted avg,0.507556,0.280000,0.204335,50.00


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [12]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


,record_id,source_index,actual_is_cancelled,predicted_is_cancelled,cancel_probability,model,threshold
0,test_0001,28,0,1,0.576120,Logistic Regression,0.2
1,test_0002,146,0,1,0.462559,Logistic Regression,0.2
2,test_0003,211,0,1,0.406685,Logistic Regression,0.2
3,test_0004,125,0,1,0.829364,Logistic Regression,0.2
4,test_0005,77,0,1,0.883050,Logistic Regression,0.2


## 11. LLM 코드 검토 체크리스트


In [13]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


,check_item,status
0,completed와 cancelled만 사용해 이진 타깃을 만들었는가?,□
1,refunded 등 다른 상태를 0 클래스에 섞지 않았는가?,□
2,예측 시점과 feature 가용성 가정을 설명할 수 있는가?,□
3,line_total = quantity × unit_price를 검증했는가?,□
4,병합에 validate를 사용하고 미매칭 0건을 확인했는가?,□
5,"order_status, target, ID, 사후 정보를 feature에서 제외했는가?",□
6,"train, validation, test를 분리했는가?",□
7,모델과 threshold는 validation에서 선택했는가?,□
8,모델과 threshold를 고정한 뒤 test를 한 번만 사용했는가?,□
9,Dummy 기준 모델과 비교했는가?,□


체크리스트 확인 (추가)
- [x] completed와 cancelled만 사용해 이진 타깃을 만들었습니다.
- [x] refunded 상태를 분류 대상에서 제외했습니다.
- [x] 예측 시점과 feature 사용 가능성을 확인했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 병합에 `validate`와 `indicator`를 사용했습니다.
- [x] 병합 전후 행 수와 미매칭 건수를 확인했습니다.
- [x] `order_status`, `is_cancelled`, 식별자, 사후 정보를 feature에서 제외했습니다.
- [x] Train, Validation, Test를 분리했습니다.
- [x] 모델과 threshold를 Validation에서 선택했습니다.
- [x] 모델과 threshold를 고정한 뒤 Test를 최종 평가했습니다.
- [x] Dummy baseline과 Logistic Regression, Random Forest를 비교했습니다.
- [x] Accuracy, Precision, Recall, F1을 함께 확인했습니다.
- [x] FP와 FN의 의미를 확인했습니다.
- [x] 공개 prediction에서 내부 식별자를 제외했습니다.
- [x] random split의 교육용 한계를 기록했습니다.
- [x] 모델 결과를 취소 원인으로 단정하지 않았습니다.

## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [14]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


선택 모델: Logistic Regression
선택 임계값: 0.2


,evaluation_split,threshold,selection_status,accuracy,precision,recall,f1
0,test,0.2,frozen_before_test,0.28,0.244444,0.846154,0.37931


model_data_internal /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_classification_model_data_internal.csv OK
target_distribution /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_target_distribution.csv OK
feature_audit /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_feature_audit.csv OK
merge_checks /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_merge_checks.csv OK
data_quality_checks /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_data_quality_checks.csv OK
split_summary /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_split_summary.csv OK
validation_comparison /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_validation_model_comparison.csv OK
threshold_metrics /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports/ch10_validation_threshold_metrics.csv OK
test_metrics /workspaces/llm-data-anal

## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.


[ 답변]

1. order_status와 is_cancelled가 feature에서 제외되었는가? 
PASS. 누수 점검에서 제외됐고 사용 feature 목록에 없습니다.

2. 병합에 validate와 indicator가 사용되었는가? 
PASS. one_to_one·many_to_one 관계를 검증했고 미매칭은 0건입니다.

3. train, validation, test가 분리되었는가? 
PASS. Train 148건, Validation 50건, Test 50건이며 각 분할에 두 클래스가 있습니다.

4. 모델과 임계값은 validation에서 선택했는가? 
PASS. Logistic Regression과 threshold 0.20을 Validation에서 선택했습니다.

5. test는 최종 평가에 한 번만 사용했는가? 
PASS. 선택을 고정한 뒤 Test를 최종 평가에 사용했습니다.

6. Dummy 기준 모델보다 나은가? 
취소 탐지 기준으로 개선됐습니다. Logistic Regression recall 0.5385, F1 0.4516이 Dummy의 0보다 높지만 accuracy는 낮습니다.

7. accuracy, precision, recall, f1을 함께 확인했는가? 
PASS. Final Test는 accuracy 0.2800, precision 0.2444, recall 0.8462, F1 0.3793입니다.

8. 예측 결과에 개인정보가 포함되지 않았는가? 
PASS. 공개 결과에서 source_index와 원본 식별자를 제외했습니다.

9. 모델 결과를 취소 원인으로 단정하지 않았는가? 
PASS. 예측 패턴으로만 해석하고 원인이나 인과관계로 단정하지 않았습니다.

권장 개선: Test 50건만으로 일반화를 단정할 수 없으므로 시간순 검증과 실제 FP/FN 비용 확인이 필요합니다. FP가 34건으로 많아 현재 모델은 운영 사용을 보류합니다.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.


## 15. 과제 답변 및 최종 판단 (추가)

### 1. 분류 문제와 입력 정보
주문 1건의 취소 여부를 예측하며 completed=0, cancelled=1로 두고 refunded 52건은 제외했습니다. 분류 대상은 완료 184건과 취소 64건, 총 248건입니다. 결과 상태나 식별자처럼 예측 이후 알게 되거나 누수 위험이 있는 정보는 feature에서 제외했습니다. 주문 상세 집계값이 예측 시점에 확보된다는 가정은 실제 운영 전에 확인해야 합니다.

### 2. 데이터 품질과 날짜 역전 처리
주문 상세 764행의 line_total 계산 불일치는 0건이고 두 병합은 248행에서 248행으로 보존됐으며 미매칭은 각각 0건입니다. 날짜 역전 분류 주문 45건은 삭제하지 않고 days_since_signup만 NaN으로 처리했으며 분석 대상은 처리 전후 248건입니다. 중앙값 대체는 숫자형 Pipeline 안에서 학습 데이터 기준으로 수행합니다. 원본 CSV는 수정하지 않았습니다.

### 3. 클래스 분포와 Validation 모델 비교
완료 주문은 184건(74.19%), 취소 주문은 64건(25.81%)입니다. Validation에서 Dummy는 accuracy 0.74였지만 취소 recall과 F1은 0이었습니다. Logistic Regression은 accuracy 0.66, precision 0.3889, recall 0.5385, F1 0.4516으로 취소 탐지 F1이 가장 높았습니다. Random Forest의 F1은 0.2609였습니다.

### 4. 데이터 분할과 Threshold
Train 148건(완료 110, 취소 38), Validation 50건(37, 13), Test 50건(37, 13)입니다. Validation에서 Logistic Regression과 threshold 0.20을 선택했습니다. 선택 이유는 Validation F1 0.4561과 취소 recall 1.0000이며, threshold 0.65의 F1 0.4545와 차이는 작습니다.

### 5. Final Test와 오류 해석
Final Test 결과는 accuracy 0.2800, precision 0.2444, recall 0.8462, F1 0.3793입니다. 혼동행렬은 TN 3, FP 34, FN 2, TP 11입니다. 취소를 놓치는 비율은 낮았지만 완료 주문 오탐이 많습니다. 실제 FP/FN 비용은 제공되지 않아 어느 오류가 더 큰 업무 손실인지 단정할 수 없습니다.

### 6. 최종 판단과 한계
현재 사용은 보류합니다. Final Test 성능이 낮고 FP가 많으며 테스트 표본은 50건입니다. 더 긴 기간의 시간순 검증, 실제 FP/FN 비용, 예측 시점의 feature 가용성을 확인해야 합니다. 결과는 예측 패턴이며 취소 원인을 설명하지 않습니다.

### 7. Validation Evidence, LLM 활용과 재현성
전체 분류 파이프라인 함수 실행은 오류 없이 끝났고 생성된 산출물 경로는 모두 OK였습니다. LLM은 날짜 이상 처리와 코드 흐름 검토에 보조적으로 사용했으며, 방법 1은 교수님 안내와 실행 결과를 대조해 반영했습니다. 별도 CLI 스크립트 실행은 현재 노트북 출력만으로 확인되지 않으므로 실행 완료로 기록하지 않습니다. Evidence 이미지는 기존 Markdown 링크 위치에 사용자가 직접 캡처해 추가합니다.

## 16. 날짜 역전 처리 상세 (추가)

- 원인: 수업용 합성 데이터 생성 과정에서 가입일과 주문일을 독립적으로 생성해 생긴 문제이며 학생의 분석 오류가 아닙니다.
- 대상: 전체 날짜 역전 56건 중 분류 대상은 45건(완료 35건, 취소 10건)입니다. 환불 11건은 이진 분류에서 제외했습니다.
- 방법: 교수님 권장 방법 1을 적용했습니다. 주문 행은 유지하고 해당 행의 days_since_signup만 NaN으로 바꿨습니다.
- 변경과 검증: 분류 데이터 생성 함수의 음수 날짜 검증을 결측 처리로 바꾸고 품질 점검표에 45건을 기록했습니다. 분류 행 수는 처리 전후 모두 248건이며 원본 CSV는 수정하지 않았습니다.
- 결측 대체: 숫자형 Pipeline의 중앙값 대체가 학습 데이터에서 학습되어 적용됩니다. 결측 처리된 45건의 실제 가입 경과 기간을 복구한 것은 아니므로 해당 feature 정보는 손실됩니다.
- 재실행 상태: Notebook의 전체 파이프라인 함수는 정상 실행되고 산출물 파일이 생성됐습니다. 별도 CLI 명령 실행 여부는 업로드된 Notebook에서 확인되지 않습니다.

# Chapter 10 답안 양식. 분류 분석으로 주문 취소 여부 예측하기

> 이 내용을 `chapter10.ipynb`의 Markdown 셀로 작성합니다. 실제 실행 결과를 근거로 작성하고, 예측 패턴을 원인처럼 단정하지 않습니다.

## 제출 정보
- 이름: `이경석`
- GitHub ID: `leeks2026`
- 작성일: `20261112`
- 최종 Notebook URL: `https://github.com/leeks2026/llm-data-analysis-study/blob/main/chapter10/chapter10.ipynb`

## 1. 분류 문제 정의
- 타깃:
  - 답: 주문 한 건의 취소 여부를 나타내는 is_cancelled입니다.
- 0의 의미:
  - 답: 완료 주문(completed)입니다.
- 1의 의미:
  - 답: 취소 주문(cancelled)입니다.
- 제외한 상태:
  - 답: refunded 52건과 기타 상태입니다. 이번 실행 데이터에서 타깃 이외 상태는 refunded 52건입니다.
- 예측 단위:
  - 답: 주문 한 건입니다.
- 예측 시점:
  - 답: 주문이 접수되고 주문 특징을 확인할 수 있지만 완료·취소 결과는 확정되지 않은 시점이라고 가정합니다.
- 예측 시점에 사용할 수 있는 정보:
  - 답: 나이, 성별, 도시, 결제수단, 주문 월·요일, 주문 상세에서 계산한 상품 수·수량·주문 금액, 가입 경과 일수입니다. 실제 운영에서 이 시점에 확보되는지는 별도 확인이 필요합니다.
- 예측 이후에만 알 수 있어 제외한 정보:
  - 답: order_status와 is_cancelled, 주문·고객·상품 식별자, 취소 사유와 취소 시각 등입니다.

### 나의 해석과 판단
완료와 취소를 비교하는 이진 분류이므로 completed=0, cancelled=1로 정하고 환불은 다른 상태이므로 제외했습니다. 예측 시점에 실제로 사용할 수 있는 정보만 입력한다는 가정을 두어 결과 상태와 식별자 같은 누수 가능 변수를 배제했습니다. 주문 상세 특징을 이 시점에 알 수 있다는 점은 교육용 가정이며 운영 전 확인이 필요합니다.

---

## 2. Feature Contract와 Leakage Audit

### 사용한 숫자형 Feature
age, item_count, total_quantity, order_amount, order_month, order_dayofweek, days_since_signup입니다. 날짜 역전 45건은 days_since_signup만 결측 처리했으며 숫자형 Pipeline의 중앙값 대체 대상입니다.

### 사용한 범주형 Feature
gender, city, payment_method입니다. 결측 대체와 OneHotEncoder는 Pipeline 안에서 학습 데이터 기준으로 처리합니다.

### 금지 Feature

```text
order_status
is_cancelled
order_id
customer_id
product_id
취소 이후 정보
```

### 나의 해석과 판단
데이터에 기록되어 있어도 예측 시점 이후에 생성되는 값이면 실제 예측에서는 알 수 없고, 학습에 넣으면 미래 정보를 미리 사용하게 됩니다. 식별자는 주문 자체의 패턴을 외우게 할 수 있어 일반화에 도움이 되지 않으므로 feature에서 제외했습니다.

---

## 3. 주문 단위 특징과 병합 검증

### 계산 관계 확인

```text
line_total = quantity × unit_price
```

- 계산 불일치 건수:
  - 답: 0건입니다.
- 주문 특징 `order_id` 중복 건수:
  - 답: 0건입니다. 주문 단위 특징 테이블은 order_id당 한 행입니다.

### 병합 검증

| 병합 | 관계 | 병합 전 행 수 | 병합 후 행 수 | 미매칭 | 판정 |
| --- | --- | ---: | ---: | ---: | --- |
| 주문 ↔ 주문 특징 | one_to_one | 248 | 248 | 0 | PASS |
| 주문 ↔ 고객 | many_to_one | 248 | 248 | 0 | PASS |

### 결과 관찰
주문 상세 764행의 line_total 계산 불일치는 0건입니다. 분류 대상 248건을 주문 특징 및 고객과 병합한 뒤에도 각각 248행이 유지됐고 미매칭은 모두 0건입니다.

### 나의 해석과 판단
미매칭은 연결 키 오류나 누락일 수 있어 0으로 채우면 실제로 확인되지 않은 주문 특징을 정상적인 0으로 오해할 수 있습니다. 이번 실행은 두 병합 모두 미매칭 0건으로 확인했습니다.

---

## 4. 클래스 비율

| 클래스 | 의미 | 개수 | 비율 |
| ---: | --- | ---: | ---: |
| 0 | completed | 184 | 74.19% |
| 1 | cancelled | 64 | 25.81% |

![클래스 비율](images/step02_class_ratio.png)

### 결과 관찰
완료 주문은 184건(74.19%), 취소 주문은 64건(25.81%)으로 완료 클래스가 더 많습니다.

### 나의 해석과 판단
다수 클래스인 완료만 예측해도 정확도가 높게 나올 수 있습니다. Dummy 모델은 accuracy 0.74였지만 취소 recall과 F1은 0이었습니다. 따라서 정확도만으로 취소 탐지 성능을 판단하지 않았습니다.

---

## 5. Dummy Baseline과 Validation 모델 비교

| 모델 | Accuracy | Precision | Recall | F1 |
| --- | ---: | ---: | ---: | ---: |
| Dummy Most Frequent | 0.7400 | 0.0000 | 0.0000 | 0.0000 |
| Logistic Regression | 0.6600 | 0.3889 | 0.5385 | 0.4516 |
| Random Forest | 0.6600 | 0.3000 | 0.2308 | 0.2609 |

![Baseline 비교](images/step03_baseline.png)

- Validation에서 선택한 모델:
  - 답: Logistic Regression입니다.

### 결과 관찰
Logistic Regression은 Dummy보다 accuracy가 낮았지만 취소 recall은 0에서 0.5385, F1은 0에서 0.4516으로 높았습니다. Random Forest의 F1은 0.2609였습니다.

### 나의 해석과 판단
취소 탐지 recall과 F1에서 Logistic Regression이 Dummy보다 나았습니다. 다만 accuracy는 Dummy 0.74, Logistic Regression 0.66이므로 개선은 취소 탐지 관점에 한정됩니다.

### 한계와 추가 확인 사항
Validation은 50건(취소 13건)뿐이라 지표 변동이 클 수 있습니다. FP/FN 비용을 확인하고 더 긴 기간 데이터로 재평가해야 합니다.

---

## 6. Train / Validation / Test 역할

| Split | 역할 | 클래스 0 건수 | 클래스 1 건수 |
| --- | --- | ---: | ---: |
| Train | 모델 학습 | 110 | 38 |
| Validation | 모델·Threshold 선택 | 37 | 13 |
| Test | 선택 이후 최종 평가 | 37 | 13 |

### 나의 해석과 판단
Test를 반복해 모델이나 임계값을 고르면 Test 정보에 맞춘 선택이 되어 최종 성능을 낙관적으로 평가할 수 있습니다. 이번 실행은 Validation에서 선택한 뒤 Test를 평가했습니다.

### 교육용 random split의 한계
무작위 분할은 과거 주문으로 미래 주문을 예측하는 시간 구조를 보장하지 않습니다. 운영 전에는 날짜 기준의 과거 학습·미래 평가(out-of-time) 검증을 추가해야 합니다.

---

## 7. Validation 성능과 우선 지표

- Accuracy:
  - 답: 0.6600
- Precision:
  - 답: 0.3889
- Recall:
  - 답: 0.5385
- F1:
  - 답: 0.4516

### 내가 가장 중요하게 본 지표
F1을 우선 지표로 보고 recall도 함께 확인했습니다.

### 그 이유
F1은 precision과 recall을 함께 반영합니다. 실제 FP/FN 비용 정보가 없어 업무상 최적 지표를 확정할 수는 없습니다.

> 실제 FP/FN 비용 정보가 없다면 가정임을 명시하세요.

---

## 8. Threshold 비교

| Threshold | Precision | Recall | F1 | 예상 특징 |
| ---: | ---: | ---: | ---: | --- |
| 0.20 | 0.2955 | 1.0000 | 0.4561 | 취소를 모두 찾았지만 FP가 늘 수 있음 |
| 0.50 | 0.3889 | 0.5385 | 0.4516 | 정밀도와 재현율의 절충 |
| 0.65 | 0.5556 | 0.3846 | 0.4545 | 정밀도는 높지만 취소를 더 놓칠 수 있음 |

![Threshold 비교](images/step05_threshold.png)

### 내가 선택한 Threshold
0.20

### 선택 이유
Validation에서 F1이 가장 높았고(0.4561), 취소 recall은 1.0000이었습니다. threshold 0.65의 F1 0.4545와 차이가 작아 업무상 최선이라고 단정할 수는 없습니다.

### 나의 해석과 판단
Test를 선택에 사용하면 최종 평가 자료에 과적합됩니다. 모델과 threshold는 Validation에서 고정했습니다.

---

## 9. FP/FN 해석

- FP 의미:
  - 답: 실제 완료 주문을 취소 위험으로 잘못 예측한 경우이며 Test에서 34건입니다.
- FN 의미:
  - 답: 실제 취소 주문을 완료로 잘못 예측한 경우이며 Test에서 2건입니다.
- 현재 목적에서 더 부담스러울 수 있는 오류:
  - 답: 취소 탐지 목적상 FN은 취소를 놓치는 오류지만 Test FP가 34건으로 많습니다. 실제 비용 정보가 없어 어느 쪽이 더 부담스러운지 확정할 수 없습니다.
- 그 판단에 추가로 필요한 업무 정보:
  - 답: 오탐으로 생기는 확인·보류·고객 응대 비용과 취소를 놓쳤을 때의 재고·배송·고객지원 비용이 필요합니다.

![Confusion Matrix](images/step06_confusion_matrix.png)

### 업무·분석적 의미
threshold 0.20은 취소 recall을 높였지만 완료 주문 37건 중 34건을 취소로 잘못 표시했습니다. 자동 조치로 연결하기에는 오탐 위험이 큽니다.

### 한계와 추가 확인 사항
Test가 50건이고 취소는 13건이라 다른 기간에도 결과가 유지되는지 알 수 없습니다. FP/FN 비용, 예측 시점의 feature 가용성, 시간에 따른 성능 변화를 추가 확인해야 합니다.

---

## 10. Final Test

- 고정한 모델:
  - 답: Logistic Regression
- 고정한 Threshold:
  - 답: 0.20
- Accuracy:
  - 답: 0.2800
- Precision:
  - 답: 0.2444
- Recall:
  - 답: 0.8462
- F1:
  - 답: 0.3793
- Validation 대비 변화:
  - 답: threshold 0.20 기준으로 accuracy 0.38→0.28, precision 0.2955→0.2444, recall 1.0000→0.8462, F1 0.4561→0.3793으로 모두 낮아졌습니다.

### 결과 관찰
Test 혼동행렬은 TN 3, FP 34, FN 2, TP 11입니다. 취소 recall은 높지만 완료 주문 오탐이 많고, Test F1 0.3793은 Validation threshold 기준 F1 0.4561보다 낮습니다.

### 나의 해석과 판단
현재 Test 결과만으로 안정적인 일반화를 확인하기 어렵습니다. Test accuracy 0.28, precision 0.2444이고 FP가 34건이라 운영 적용 근거가 부족합니다. Test를 보고 모델이나 threshold를 다시 고르지는 않았습니다.

### 선택 보호 확인
- [x] Test 결과를 보기 전에 모델을 고정했습니다.
- [x] Test 결과를 보기 전에 Threshold를 고정했습니다.
- [x] Test 결과를 본 뒤 같은 Test를 이용해 선택을 바꾸지 않았습니다.

---

## 11. Internal / Public 결과 구분

### 공개 Prediction 컬럼

```text
record_id
actual_is_cancelled
predicted_is_cancelled
cancel_probability
model
threshold
```

### 공개 결과에 없어야 할 컬럼

```text
source_index
order_id
customer_id
product_id
```

### 결과 관찰
공개 Prediction은 record_id, actual_is_cancelled, predicted_is_cancelled, cancel_probability, model, threshold만 사용합니다. 내부 진단용 표에는 source_index가 포함될 수 있으며 공개 결과 파일은 분리 저장합니다.

### 나의 해석과 판단
내부 연결 키는 오류 추적에 유용할 수 있지만 공개에는 불필요한 식별·연결 단서가 될 수 있습니다.

---

## 12. Validation Evidence

다음 Evidence 중 확인한 항목에 체크합니다.

- [x] `ch10_target_distribution.csv`
- [x] `ch10_feature_audit.csv`
- [x] `ch10_merge_checks.csv`
- [x] `ch10_data_quality_checks.csv`
- [x] `ch10_split_summary.csv`
- [x] `ch10_validation_model_comparison.csv`
- [x] `ch10_validation_threshold_metrics.csv`
- [x] `ch10_test_metrics.csv`
- [x] `ch10_confusion_matrix.csv`
- [x] `ch10_classification_validation.csv`

### 자동 Validation 결과
전체 분류 파이프라인이 오류 없이 완료됐고 산출물 경로가 모두 OK였습니다. check별 상태는 생성된 ch10_classification_validation.csv에서 확인합니다.

### 자동 PASS가 의미하지 않는 것
운영 적합성, 공정성, 시간에 따른 성능 변화 및 실제 FP/FN 비용은 자동 검증만으로 보장되지 않습니다.

---

## 13. 최종 사용 판단

- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. Test accuracy 0.28, precision 0.2444, F1 0.3793으로 성능이 낮습니다.
2. 혼동행렬에서 FP가 34건으로 많습니다.
3. Test 50건만으로 다른 기간의 안정성과 실제 업무 비용을 판단할 수 없습니다.

### 현재 모델의 가장 큰 위험
완료 주문을 취소 위험으로 잘못 예측하는 FP가 많아 자동 조치 시 정상 주문에 불필요한 확인이나 제한이 생길 수 있습니다.

### 다음 개선 우선순위
1. 예측 시점의 feature 가용성과 합성 데이터 생성 규칙을 확인합니다.
2. 더 긴 기간의 데이터를 확보해 시간순 검증과 out-of-time 테스트를 합니다.
3. 실제 FP/FN 비용을 확인해 Validation 목표 지표와 threshold를 정합니다.

### 예측과 인과 구분
취소 recall은 높지만 정상 완료 주문 오탐도 많다는 예측 패턴을 보였습니다. 이 결과만으로 고객 특성이나 결제수단이 취소의 원인이라고 주장할 수 없습니다.

---

## 최종 체크

- [x] `completed=0`, `cancelled=1` 타깃 범위를 정의했습니다.
- [x] `refunded`와 기타 상태를 제외했습니다.
- [x] 예측 시점과 Feature Contract를 정의했습니다.
- [x] target, ID, 사후 정보를 feature에서 제외했습니다.
- [x] `line_total = quantity × unit_price`를 검증했습니다.
- [x] 주문 단위 특징과 병합 관계를 검증했습니다.
- [x] 클래스 비율을 확인했습니다.
- [x] Dummy baseline과 비교했습니다.
- [x] Train / Validation / Test 역할을 구분했습니다.
- [x] 모델을 Validation에서 선택했습니다.
- [x] Threshold를 Validation에서 선택했습니다.
- [x] Final Test 전에 선택을 고정했습니다.
- [x] Precision / Recall / F1을 해석했습니다.
- [x] FP/FN의 의미를 설명했습니다.
- [x] Public prediction에서 식별자를 제거했습니다.
- [x] Validation Evidence를 확인했습니다.
- [x] random split의 한계를 기록했습니다.
- [x] 예측 패턴을 원인처럼 단정하지 않았습니다.
- [x] 최종 Notebook URL을 제출합니다.

## 날짜 역전 처리 상세

- 원인: 확인 결과, 수업용 합성 데이터를 만들 때 가입일과 주문일을 독립적으로 생성해 날짜 역전이 발생했습니다. 분석 오류는 아닙니다.
- 대상: 전체 역전 56건 중 분류 대상은 45건(완료 35건, 취소 10건)이고, 나머지 환불 11건은 이진 분류에서 제외됩니다.
- 선택한 방법: 방법 1. 날짜 역전 주문은 유지하고 해당 주문의 days_since_signup만 결측 처리했습니다.
- 변경한 부분: 분류 데이터 생성 코드에서 음수 days_since_signup를 세고 NaN으로 바꿨으며, 품질 점검표에 처리 건수 45를 기록했습니다. 원본 CSV는 수정하지 않았습니다.
- 처리 전후: 분류 행 수는 처리 전 248건, 처리 후 248건입니다. 45건의 파생값만 결측이고 나머지 203건은 계산값을 유지했습니다. 중앙값 대체는 Pipeline 안에서 학습 데이터 기준으로 수행됩니다.
- 한계: 결측 처리로 실제 가입 경과 기간을 복구한 것은 아닙니다. 이 값이 없는 주문의 취소 위험은 다른 feature에 의존하며, 실제 데이터에서 가입일 정합성과 예측 시점의 feature 가용성을 다시 확인해야 합니다.
- 재실행 증거: 업로드된 Notebook에서는 전체 파이프라인 함수 실행과 산출물 생성이 확인됩니다.

## LLM 활용 및 사람의 검증 기록

- 목적: 날짜 역전 처리와 코드 흐름을 검토했습니다.
- Prompt 요약: 분류 대상 날짜 역전 45건을 유지하면서 파생 변수만 결측 처리하고 검증·Pipeline 처리·한계를 확인하도록 요청했습니다.
- 제안 및 반영: 교수님이 허용한 방법 1을 적용해 원본 CSV와 주문 행을 유지하고 days_since_signup만 결측 처리했습니다.
- 사람의 검증 근거: 실행 결과(248행 유지, 45건 처리, 중앙값 대체 Pipeline)를 대조했습니다. 모델 지표와 혼동행렬은 노트북 출력에서 확인했습니다.

## 13번 LLM 검토 프롬프트 답변

### 필수 수정 및 검토 결과
날짜 역전 45건은 삭제하지 않고 days_since_signup만 결측 처리했습니다. 분류 대상 248건은 유지됐고 completed 184건·cancelled 64건, refunded 52건 제외, line_total 계산 불일치 0건, 두 병합 미매칭 0건을 확인했습니다. 누수 가능 상태·식별자·사후 정보는 feature에서 제외했습니다.

검토 결과: train·validation·test를 분리했고 Logistic Regression과 threshold 0.20을 validation에서 선택한 뒤 test를 최종 평가에 사용했습니다. Dummy baseline과 비교하고 accuracy, precision, recall, F1, confusion matrix, FP/FN을 확인했습니다. 공개 결과에는 내부 식별자를 포함하지 않았습니다.

### 권장 개선
Test 50건만으로 일반화를 단정할 수 없으므로 더 긴 기간의 시간순 검증과 실제 FP/FN 비용 확인이 필요합니다. 모델 결과는 예측 패턴이며 취소 원인으로 해석하지 않았고 FP가 34건으로 많아 현재 모델은 운영 사용을 보류합니다.